# 🔬 Pipeline Darija — Prétraitement + Fusion + Enrichissement + Fichiers Finaux

## Étapes du pipeline :
| # | Étape | Fichier généré |
|---|---|---|
| 1 | Prétraitement JSON 1 → `entry_id, text, label, decision` | `json1_clean.csv` |
| 2 | Prétraitement JSON 2 → logique `garder_gemini` / `garder_manuel` / `corriger` | `json2_clean.csv` |
| 3 | Prétraitement JSON 3 → `entry_id, text, label, decision` | `json3_clean.csv` |
| 4 | Prétraitement JSON 4 → `entry_id, text, label, decision` | `json4_clean.csv` |
| 5 | Fusion des 4 fichiers nettoyés | `merged.csv` + `merged.json` |
| 6 | Suppression des emojis → `text_normalized` | *(en mémoire)* |
| 7 | Enrichissement avec le CSV de référence | *(en mémoire)* |
| 8 | **Fichier Contrôle Qualité** — tous les textes + colonnes QC | `quality_control.csv` |
| 9 | **Fichier Modélisation** — garder+corriger uniquement, sans `decision` | `modelisation.csv` |

---
## 📦 0. Imports & Configuration

In [1]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path

# ── Chemins des fichiers d'entrée ────────────────────────────────────────────
JSON_FILE_1   = "data/861_verifie_clean.json"
JSON_FILE_2   = "data/1000_gold_verifie.json"
JSON_FILE_3   = "data/data_verifie_auto.json"
JSON_FILE_4   = "data/data_verifie_manuel.json"
CSV_REFERENCE = "data/dataset_total_clean.csv"

# ── Fichiers de sortie ───────────────────────────────────────────────────────
OUT_JSON1_CLEAN  = "data_final/json1_clean.csv"
OUT_JSON2_CLEAN  = "data_final/json2_clean.csv"
OUT_JSON3_CLEAN  = "data_final/json3_clean.csv"
OUT_JSON4_CLEAN  = "data_final/json4_clean.csv"
OUT_MERGED_CSV   = "data_final/merged.csv"
OUT_MERGED_JSON  = "data_final/merged.json"
OUT_QC_CSV       = "data_final/quality_control.csv"   # Contrôle qualité — TOUS les textes
OUT_MODEL_CSV    = "data_final/modelisation.csv"       # Modélisation — garder + corriger uniquement

# ── Colonnes cibles finales (communes aux 4 JSON) ────────────────────────────
FINAL_COLS = ["entry_id", "text", "label", "decision"]

print("✅ Configuration OK")
print(f"   Colonnes cibles : {FINAL_COLS}")

✅ Configuration OK
   Colonnes cibles : ['entry_id', 'text', 'label', 'decision']


---
## 🔧 Fonctions utilitaires

In [2]:
def load_json(path):
    """Charge un fichier JSON (liste ou objet unique) en liste de dicts."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else [data]


def save_csv(df, path, label=""):
    """Sauvegarde un DataFrame en CSV UTF-8 avec BOM."""
    df.to_csv(path, index=False, encoding="utf-8-sig")
    tag = f"[{label}] " if label else ""
    print(f"  ✅ {tag}Sauvegardé : {path}  ({len(df)} lignes × {len(df.columns)} colonnes)")


def preview(df, n=3):
    """Affiche un aperçu rapide du DataFrame."""
    print(f"   Shape : {df.shape}  |  Colonnes : {list(df.columns)}")
    return df.head(n)


print("✅ Fonctions utilitaires chargées")

✅ Fonctions utilitaires chargées


---
## ÉTAPE 1 — Prétraitement du JSON 1
**Règle :** Garder `entry_id`, `text`, `label`, `decision` — tels quels.

In [3]:
print("═" * 55)
print("ÉTAPE 1 — Prétraitement JSON 1")
print("═" * 55)

raw1 = load_json(JSON_FILE_1)
df1  = pd.DataFrame(raw1)
print(f"  Chargé : {len(df1)} enregistrements")
print(f"  Colonnes brutes : {list(df1.columns)}")

missing = [c for c in FINAL_COLS if c not in df1.columns]
if missing:
    print(f"  ⚠️  Colonnes absentes (seront NaN) : {missing}")

df1_clean = df1.reindex(columns=FINAL_COLS)

print(f"\n  Distribution 'decision' :")
print(df1_clean['decision'].value_counts(dropna=False).to_string())
print(f"\n  Labels vides : {df1_clean['label'].isna().sum()}")

save_csv(df1_clean, OUT_JSON1_CLEAN, "JSON1")
preview(df1_clean)

═══════════════════════════════════════════════════════
ÉTAPE 1 — Prétraitement JSON 1
═══════════════════════════════════════════════════════
  Chargé : 861 enregistrements
  Colonnes brutes : ['entry_id', 'annotator', 'created_at', 'decision', 'id', 'label', 'lead_time', 'text', 'updated_at']

  Distribution 'decision' :
decision
garder      683
corriger    163
rejeter      15

  Labels vides : 0
  ✅ [JSON1] Sauvegardé : enfin/json1_clean.csv  (861 lignes × 4 colonnes)
   Shape : (861, 4)  |  Colonnes : ['entry_id', 'text', 'label', 'decision']


,entry_id,text,label,decision
0,darija_050244,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,food,garder
1,darija_012475,",شكون جاي من 2025🎉🎉🎉🎉",inclassable,corriger
2,darija_071102,فلان غزال تبارك الله ،هل يمكن تعويض الشوفان با...,food,garder


---
## ÉTAPE 2 — Prétraitement du JSON 2

| `decision` original | `label` retenu | `decision` finale |
|---|---|---|
| `garder_gemini` | `gemini_label` | `garder` |
| `garder_manuel` | `manuel_label` | `corriger` |
| `corriger` | `label_corrige` | `corriger` |
| autre | `label` existant | inchangée |

In [4]:
print("═" * 55)
print("ÉTAPE 2 — Prétraitement JSON 2")
print("═" * 55)

raw2 = load_json(JSON_FILE_2)
df2  = pd.DataFrame(raw2)
print(f"  Chargé : {len(df2)} enregistrements")
print(f"  Colonnes brutes : {list(df2.columns)}")
print(f"\n  Distribution 'decision' (original) :")
print(df2['decision'].value_counts(dropna=False).to_string())


def resolve_label_decision(row):
    decision_orig = str(row.get('decision', '')).strip().lower()

    if decision_orig == 'garder_gemini':
        label    = row.get('gemini_label', np.nan)
        decision = 'garder'

    elif decision_orig == 'garder_manuel':
        label    = row.get('manuel_label', np.nan)
        decision = 'corriger'

    elif decision_orig == 'corriger':
        label_corrige = str(row.get('label_corrige', '')).strip()
        label    = label_corrige if label_corrige else np.nan
        decision = 'corriger'

    else:
        label    = row.get('label', np.nan)
        decision = row.get('decision', np.nan)

    return pd.Series({'label': label, 'decision': decision})


resolved  = df2.apply(resolve_label_decision, axis=1)
df2_clean = pd.DataFrame({
    'entry_id': df2['entry_id'],
    'text'    : df2['text'],
    'label'   : resolved['label'],
    'decision': resolved['decision'],
})

df2_clean['label']    = df2_clean['label'].replace('', np.nan)
df2_clean['decision'] = df2_clean['decision'].replace('', np.nan)

print(f"\n  Distribution 'decision' (après transformation) :")
print(df2_clean['decision'].value_counts(dropna=False).to_string())
print(f"\n  Labels vides : {df2_clean['label'].isna().sum()}")

save_csv(df2_clean, OUT_JSON2_CLEAN, "JSON2")
preview(df2_clean)

═══════════════════════════════════════════════════════
ÉTAPE 2 — Prétraitement JSON 2
═══════════════════════════════════════════════════════
  Chargé : 1000 enregistrements
  Colonnes brutes : ['entry_id', 'annotator', 'created_at', 'decision', 'gemini_label', 'id', 'label_corrige', 'lead_time', 'manuel_label', 'text', 'updated_at']

  Distribution 'decision' (original) :
decision
garder_gemini    795
garder_manuel    195
corriger          10

  Distribution 'decision' (après transformation) :
decision
garder      795
corriger    205

  Labels vides : 0
  ✅ [JSON2] Sauvegardé : enfin/json2_clean.csv  (1000 lignes × 4 colonnes)
   Shape : (1000, 4)  |  Colonnes : ['entry_id', 'text', 'label', 'decision']


,entry_id,text,label,decision
0,darija_088097,اسعار غالية,politics,garder
1,darija_086479,وصلتي حتي لهنا بغيت لميمتك قصر فالجنة الفردوس ...,publicite,garder
2,darija_087536,اه متافقة معاك انا دويت لك علي الولدات والكدوب...,opinion,garder


---
## ÉTAPE 3 — Prétraitement du JSON 3
**Règle :** Garder `entry_id`, `text`, `label`, `decision`. Fallback `gemini_label` si `label` absent.

In [5]:
print("═" * 55)
print("ÉTAPE 3 — Prétraitement JSON 3")
print("═" * 55)

raw3 = load_json(JSON_FILE_3)
df3  = pd.DataFrame(raw3)
print(f"  Chargé : {len(df3)} enregistrements")
print(f"  Colonnes brutes : {list(df3.columns)}")

if 'label' not in df3.columns:
    df3['label'] = np.nan

if 'gemini_label' in df3.columns:
    mask = df3['label'].isna() | (df3['label'].astype(str).str.strip() == '')
    df3.loc[mask, 'label'] = df3.loc[mask, 'gemini_label']
    print(f"  ℹ️  Fallback gemini_label → label appliqué sur {mask.sum()} lignes")

df3_clean = df3.reindex(columns=FINAL_COLS)

print(f"\n  Distribution 'decision' :")
print(df3_clean['decision'].value_counts(dropna=False).to_string())
print(f"\n  Labels vides : {df3_clean['label'].isna().sum()}")

save_csv(df3_clean, OUT_JSON3_CLEAN, "JSON3")
preview(df3_clean)

═══════════════════════════════════════════════════════
ÉTAPE 3 — Prétraitement JSON 3
═══════════════════════════════════════════════════════
  Chargé : 14300 enregistrements
  Colonnes brutes : ['entry_id', 'text', 'gemini_label', 'label', 'decision']
  ℹ️  Fallback gemini_label → label appliqué sur 0 lignes

  Distribution 'decision' :
decision
garder      11635
corriger     2535
rejeter       130

  Labels vides : 0
  ✅ [JSON3] Sauvegardé : enfin/json3_clean.csv  (14300 lignes × 4 colonnes)
   Shape : (14300, 4)  |  Colonnes : ['entry_id', 'text', 'label', 'decision']


,entry_id,text,label,decision
0,darija_116902,واحد الكاليتي فالانجينيور : كيسمع بتمعن مفيهش ...,opinion,garder
1,darija_087374,هذا راه كاين فالمجتمع الغربي ومقنن وهو انفصال ...,politics,garder
2,darija_091635,Khoya mstafa ana mea had l9adya hitach mn lhwa...,politics,garder


---
## ÉTAPE 4 — Prétraitement du JSON 4
**Règle :** Garder `entry_id`, `text`, `label`, `decision`. Le label annoté est directement dans `label`.

In [6]:
print("═" * 55)
print("ÉTAPE 4 — Prétraitement JSON 4")
print("═" * 55)

raw4 = load_json(JSON_FILE_4)
df4  = pd.DataFrame(raw4)
print(f"  Chargé : {len(df4)} enregistrements")
print(f"  Colonnes brutes : {list(df4.columns)}")

missing = [c for c in FINAL_COLS if c not in df4.columns]
if missing:
    print(f"  ⚠️  Colonnes absentes (seront NaN) : {missing}")

df4_clean = df4.reindex(columns=FINAL_COLS)
df4_clean['label']    = df4_clean['label'].replace('', np.nan)
df4_clean['decision'] = df4_clean['decision'].replace('', np.nan)

print(f"\n  Distribution 'decision' :")
print(df4_clean['decision'].value_counts(dropna=False).to_string())
print(f"\n  Distribution 'label' (top 10) :")
print(df4_clean['label'].value_counts(dropna=False).head(10).to_string())
print(f"\n  Labels vides : {df4_clean['label'].isna().sum()}")

save_csv(df4_clean, OUT_JSON4_CLEAN, "JSON4")
preview(df4_clean)

═══════════════════════════════════════════════════════
ÉTAPE 4 — Prétraitement JSON 4
═══════════════════════════════════════════════════════
  Chargé : 8641 enregistrements
  Colonnes brutes : ['text', 'entry_id', 'gemini_label', 'id', 'label', 'decision', 'annotator', 'annotation_id', 'created_at', 'updated_at', 'lead_time']

  Distribution 'decision' :
decision
garder      6865
corriger    1574
rejeter      202

  Distribution 'label' (top 10) :
label
soutien             1879
opinion             1554
inclassable         1017
question_general     830
food                 785
politics             581
entertainment        508
business             474
sports               341
medical              261

  Labels vides : 2
  ✅ [JSON4] Sauvegardé : enfin/json4_clean.csv  (8641 lignes × 4 colonnes)
   Shape : (8641, 4)  |  Colonnes : ['entry_id', 'text', 'label', 'decision']


,entry_id,text,label,decision
0,darija_006170,2022.6.15 knw3edkom men hna l 10 ans radi t3rf...,inclassable,corriger
1,darija_001540,خويا سيمو ممكن نمرة تاعك انا بديت فهاد ضومين ب...,question_general,corriger
2,darija_046263,واش ممكن منديروش شعرية روز نديرو فقط دجاج والبصل,food,garder


---
## ÉTAPE 5 — Fusion des 4 fichiers nettoyés
Déduplication sur `entry_id` — priorité aux entrées avec label renseigné.

In [7]:
print("═" * 55)
print("ÉTAPE 5 — Fusion")
print("═" * 55)

print(f"  JSON1 : {len(df1_clean):>6} lignes")
print(f"  JSON2 : {len(df2_clean):>6} lignes")
print(f"  JSON3 : {len(df3_clean):>6} lignes")
print(f"  JSON4 : {len(df4_clean):>6} lignes")

df1_clean = df1_clean.copy(); df1_clean['_source'] = 'json1'
df2_clean = df2_clean.copy(); df2_clean['_source'] = 'json2'
df3_clean = df3_clean.copy(); df3_clean['_source'] = 'json3'
df4_clean = df4_clean.copy(); df4_clean['_source'] = 'json4'

df_all = pd.concat([df1_clean, df2_clean, df3_clean, df4_clean], ignore_index=True)
print(f"\n  Total avant déduplication : {len(df_all)}")

df_all['_label_missing'] = df_all['label'].isna()
df_all = (df_all
          .sort_values('_label_missing')
          .drop_duplicates(subset='entry_id', keep='first')
          .drop(columns=['_label_missing'])
          .reset_index(drop=True))

print(f"  Total après déduplication  : {len(df_all)}")
print(f"\n  Distribution 'decision' finale :")
print(df_all['decision'].value_counts(dropna=False).to_string())
print(f"\n  Distribution 'label' finale (top 10) :")
print(df_all['label'].value_counts(dropna=False).head(10).to_string())
print(f"\n  Répartition par source :")
print(df_all['_source'].value_counts().to_string())

df_merged = df_all.drop(columns=['_source'])

save_csv(df_merged, OUT_MERGED_CSV, "MERGED")

with open(OUT_MERGED_JSON, "w", encoding="utf-8") as f:
    json.dump(df_merged.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
print(f"  ✅ [MERGED] Sauvegardé : {OUT_MERGED_JSON}")

preview(df_merged)

═══════════════════════════════════════════════════════
ÉTAPE 5 — Fusion
═══════════════════════════════════════════════════════
  JSON1 :    861 lignes
  JSON2 :   1000 lignes
  JSON3 :  14300 lignes
  JSON4 :   8641 lignes

  Total avant déduplication : 24802
  Total après déduplication  : 24800

  Distribution 'decision' finale :
decision
garder      19977
corriger     4476
rejeter       347

  Distribution 'label' finale (top 10) :
label
opinion             5203
soutien             5062
food                2229
inclassable         2067
question_general    2005
politics            1854
entertainment       1734
business            1493
sports               996
medical              829

  Répartition par source :
_source
json3    14299
json4     8640
json2     1000
json1      861
  ✅ [MERGED] Sauvegardé : enfin/merged.csv  (24800 lignes × 4 colonnes)
  ✅ [MERGED] Sauvegardé : enfin/merged.json
   Shape : (24800, 4)  |  Colonnes : ['entry_id', 'text', 'label', 'decision']


,entry_id,text,label,decision
0,darija_050244,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,food,garder
1,darija_082567,مالكم.درقو.فالبنت.واش.حنا.خصنا,inclassable,corriger
2,darija_006392,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,business,garder


---
## ÉTAPE 6 — Suppression des Emojis → `text_normalized`

On crée la colonne `text_normalized` = `text` nettoyé de tous les emojis.  
Le `text` original est **conservé intact**.

Types d'emojis couverts : 😀 emoticons · 🇲🇦 drapeaux · 👋🏽 modificateurs peau · ❤️ variantes · 👨‍💻 séquences ZWJ · symboles divers

In [8]:
print("═" * 55)
print("ÉTAPE 6 — Suppression des emojis → text_normalized")
print("═" * 55)

# ── Regex Unicode complète pour tous les types d'emojis ──────────────────────
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"   # Emoticons
    "\U0001F300-\U0001F5FF"   # Symboles & pictogrammes divers
    "\U0001F680-\U0001F6FF"   # Transport & cartes
    "\U0001F700-\U0001F77F"   # Symboles alchimiques
    "\U0001F780-\U0001F7FF"   # Géométriques étendus
    "\U0001F800-\U0001F8FF"   # Flèches C
    "\U0001F900-\U0001F9FF"   # Pictogrammes supplémentaires A
    "\U0001FA00-\U0001FA6F"   # Jeux d'échecs
    "\U0001FA70-\U0001FAFF"   # Pictogrammes supplémentaires B
    "\U00002702-\U000027B0"   # Dingbats
    "\U000024C2-\U0001F251"   # Symboles divers
    "\U0001F004-\U0001F0CF"   # Mahjong, cartes
    "\U0001F170-\U0001F251"   # Symboles de groupe
    "\U00002500-\U00002BEF"   # Caractères techniques
    "\U00010000-\U0010FFFF"   # Suppléments SMP
    "\uFE0F"                  # Variation selector-16
    "\u200D"                  # Zero-width joiner
    "\u20E3"                  # Combining enclosing keycap
    "]+",
    flags=re.UNICODE
)


def remove_emojis(value):
    """Supprime tous les emojis d'un texte et nettoie les espaces résiduels."""
    if not isinstance(value, str):
        return value
    cleaned = EMOJI_PATTERN.sub("", value)
    cleaned = re.sub(r" {2,}", " ", cleaned).strip()
    return cleaned


# ── Test de la fonction ───────────────────────────────────────────────────────
test = ",شكون جاي من 2025🎉🎉🎉🎉"
print(f"  Avant : {test}")
print(f"  Après : {remove_emojis(test)}")
print()

# ── Création de text_normalized dans df_merged ────────────────────────────────
df_merged['text_normalized'] = df_merged['text'].apply(remove_emojis)

# ── Statistiques de nettoyage ────────────────────────────────────────────────
n_with_emoji = (df_merged['text'] != df_merged['text_normalized']).sum()
total_chars_removed = (
    df_merged['text'].astype(str).str.len().sum() -
    df_merged['text_normalized'].astype(str).str.len().sum()
)

print(f"  Textes contenant des emojis : {n_with_emoji} / {len(df_merged)}")
print(f"  Total caractères emoji supprimés : {total_chars_removed}")
print(f"\n  Exemple avant/après :")

exemples = df_merged[df_merged['text'] != df_merged['text_normalized']][['text', 'text_normalized']].head(3)
for _, row in exemples.iterrows():
    print(f"    Avant : {row['text'][:80]}")
    print(f"    Après : {row['text_normalized'][:80]}")
    print()

═══════════════════════════════════════════════════════
ÉTAPE 6 — Suppression des emojis → text_normalized
═══════════════════════════════════════════════════════
  Avant : ,شكون جاي من 2025🎉🎉🎉🎉
  Après : ,شكون جاي من 2025

  Textes contenant des emojis : 6592 / 24800
  Total caractères emoji supprimés : 22827

  Exemple avant/après :
    Avant : بصحة ❤خويا ولاكن انا عندي خبر وبسي يستاهل غير 21صراحة اخي
    Après : بصحة خويا ولاكن انا عندي خبر وبسي يستاهل غير 21صراحة اخي

    Avant : Mabah w shaos brojoula 👐🫡
    Après : Mabah w shaos brojoula

    Avant : هرباتني هاديك الي قال السيمو لايف كان سكران وقيلة 😂😂
    Après : هرباتني هاديك الي قال السيمو لايف كان سكران وقيلة



---
## ÉTAPE 7 — Enrichissement avec le CSV de référence

Jointure : `entry_id` (fusionné) ↔ `id` (CSV référence)

### Colonnes importées du CSV de référence :

| Colonne | Utilité principale |
|---|---|
| `darija_type` | Contrôle qualité + modélisation |
| `det_confidence` | Contrôle qualité (filtrage fiabilité) |
| `source_type` | Contrôle qualité + analyse par source |
| `domain` | Contrôle qualité |
| `channel_name` | Contrôle qualité + traçabilité |
| `published_at` | Contrôle qualité + analyse temporelle |
| `language` | Contrôle qualité (filtrage linguistique) |
| `script` | Contrôle qualité + modélisation |
| `has_codeswitching` | Contrôle qualité + modélisation |
| `has_arabizi` | Contrôle qualité + modélisation |
| `n_words` | Contrôle qualité + modélisation (features) |
| `n_chars` | Contrôle qualité + modélisation (features) |
| `n_arabic_chars` | Modélisation (features linguistiques) |
| `n_latin_chars` | Modélisation (features linguistiques) |
| `darija_ara_score` | Contrôle qualité (filtrage Darija) |
| `darija_lat_score` | Contrôle qualité (filtrage Darija) |
| `msa_score` | Contrôle qualité (exclusion MSA) |

In [9]:
print("═" * 55)
print("ÉTAPE 7 — Enrichissement CSV de référence")
print("═" * 55)

df_ref = pd.read_csv(CSV_REFERENCE, encoding="utf-8-sig", low_memory=False)
print(f"  CSV référence : {df_ref.shape[0]} lignes × {df_ref.shape[1]} colonnes")

# ── Colonnes à importer ───────────────────────────────────────────────────────
REF_COLS = [
    "id",                # clé de jointure
    "darija_type",       # type de dialecte
    "det_confidence",    # score de confiance de détection
    "source_type",       # provenance (youtube_comment, etc.)
    "domain",            # domaine thématique initial
    "channel_name",      # canal source
    "published_at",      # date de publication
    "language",          # langue détectée
    "script",            # système d'écriture
    "has_codeswitching", # alternance de codes
    "has_arabizi",       # présence d'arabizi
    "n_words",           # nombre de mots
    "n_chars",           # nombre de caractères
    "n_arabic_chars",    # caractères arabes
    "n_latin_chars",     # caractères latins
    "darija_ara_score",  # score Darija arabe
    "darija_lat_score",  # score Darija latin
    "msa_score",         # score arabe standard
]

existing = [c for c in REF_COLS if c in df_ref.columns]
absent   = [c for c in REF_COLS if c not in df_ref.columns]
if absent:
    print(f"  ⚠️  Colonnes absentes du CSV (ignorées) : {absent}")

df_ref_slim = df_ref[existing].copy()

# ── Jointure LEFT sur entry_id ↔ id ─────────────────────────────────────────
df_enriched = df_merged.merge(
    df_ref_slim,
    left_on='entry_id',
    right_on='id',
    how='left'
)

if 'id' in df_enriched.columns:
    df_enriched = df_enriched.drop(columns=['id'])

n_matched = df_enriched['source_type'].notna().sum() if 'source_type' in df_enriched.columns else 'N/A'
print(f"\n  Entrées enrichies (match trouvé) : {n_matched} / {len(df_enriched)}")
print(f"  Entrées sans correspondance      : {len(df_enriched) - (n_matched if isinstance(n_matched, int) else 0)}")
print(f"\n  Colonnes disponibles ({df_enriched.shape[1]}) :")
print(f"  {list(df_enriched.columns)}")

preview(df_enriched)

═══════════════════════════════════════════════════════
ÉTAPE 7 — Enrichissement CSV de référence
═══════════════════════════════════════════════════════
  CSV référence : 25001 lignes × 21 colonnes

  Entrées enrichies (match trouvé) : 24797 / 24800
  Entrées sans correspondance      : 24800

  Colonnes disponibles (22) :
  ['entry_id', 'text', 'label', 'decision', 'text_normalized', 'darija_type', 'det_confidence', 'source_type', 'domain', 'channel_name', 'published_at', 'language', 'script', 'has_codeswitching', 'has_arabizi', 'n_words', 'n_chars', 'n_arabic_chars', 'n_latin_chars', 'darija_ara_score', 'darija_lat_score', 'msa_score']
   Shape : (24800, 22)  |  Colonnes : ['entry_id', 'text', 'label', 'decision', 'text_normalized', 'darija_type', 'det_confidence', 'source_type', 'domain', 'channel_name', 'published_at', 'language', 'script', 'has_codeswitching', 'has_arabizi', 'n_words', 'n_chars', 'n_arabic_chars', 'n_latin_chars', 'darija_ara_score', 'darija_lat_score', 'msa_score

,entry_id,text,label,decision,text_normalized,darija_type,det_confidence,source_type,domain,channel_name,...,script,has_codeswitching,has_arabizi,n_words,n_chars,n_arabic_chars,n_latin_chars,darija_ara_score,darija_lat_score,msa_score
0,darija_050244,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,food,garder,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,darija_arabic,0.600,youtube_comment,food,Cuisine Halima Filali,...,arabic,False,False,10.0,58.0,49.0,0.0,0.2000,0.0000,0.0
1,darija_082567,مالكم.درقو.فالبنت.واش.حنا.خصنا,inclassable,corriger,مالكم.درقو.فالبنت.واش.حنا.خصنا,darija_arabic,0.500,youtube_comment,food,Simo Stars,...,arabic,False,False,1.0,30.0,25.0,0.0,0.3333,0.0000,0.0
2,darija_006392,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,business,garder,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,darija_latin,0.625,youtube_comment,business,Youssef Akalal,...,latin,False,True,12.0,69.0,0.0,57.0,0.0000,0.3333,0.0


---
## ÉTAPE 8 — Fichier Contrôle Qualité

**Contient : TOUS les textes** (garder + corriger + rejeter)

### Colonnes du fichier `quality_control.csv` :

| Colonne | Rôle dans le contrôle qualité |
|---|---|
| `entry_id` | Identifiant unique pour traçabilité |
| `text` | Texte original brut (avec emojis) |
| `text_normalized` | Texte après suppression des emojis |
| `label` | Label assigné à vérifier |
| `decision` | `garder` / `corriger` / `rejeter` — clé de QC |
| `source_type` | Permet de vérifier la provenance |
| `channel_name` | Traçabilité de la source |
| `domain` | Cohérence label ↔ domaine |
| `language` | Vérification linguistique |
| `script` | Vérification du script (arabe/latin) |
| `darija_type` | Type Darija pour validation |
| `det_confidence` | Fiabilité de la détection automatique |
| `darija_ara_score` | Score Darija arabe (filtrage MSA) |
| `darija_lat_score` | Score Darija latin |
| `msa_score` | Score MSA — textes à fort score à réviser |
| `has_codeswitching` | Textes mixtes à vérifier manuellement |
| `has_arabizi` | Arabizi → vérification orthographe |
| `n_words` | Textes trop courts/longs à signaler |
| `n_chars` | Longueur brute du texte |
| `published_at` | Date pour analyse temporelle |

In [10]:
print("═" * 55)
print("ÉTAPE 8 — Fichier Contrôle Qualité")
print("═" * 55)

# ── Colonnes pour le contrôle qualité ────────────────────────────────────────
QC_COLS = [
    # ── Identité & texte ────────────────────────────────
    "entry_id",
    "text",              # texte original (avec emojis)
    "text_normalized",   # texte sans emojis
    "label",
    "decision",          # garder / corriger / rejeter
    # ── Provenance & traçabilité ─────────────────────────
    "source_type",
    "channel_name",
    "domain",
    "published_at",
    # ── Vérification linguistique ────────────────────────
    "language",
    "script",
    "darija_type",
    "has_codeswitching",
    "has_arabizi",
    # ── Scores de qualité (filtrage MSA / Darija) ────────
    "det_confidence",
    "darija_ara_score",
    "darija_lat_score",
    "msa_score",
    # ── Métriques textuelles ─────────────────────────────
    "n_words",
    "n_chars",
]

# Ne garder que les colonnes qui existent réellement
qc_cols_present = [c for c in QC_COLS if c in df_enriched.columns]
qc_cols_absent  = [c for c in QC_COLS if c not in df_enriched.columns]

if qc_cols_absent:
    print(f"  ⚠️  Colonnes absentes (non enrichies) : {qc_cols_absent}")

df_qc = df_enriched[qc_cols_present].copy()

# ── Statistiques ─────────────────────────────────────────────────────────────
print(f"\n  Nombre total de textes : {len(df_qc)}")
print(f"\n  Distribution 'decision' :")
print(df_qc['decision'].value_counts(dropna=False).to_string())
print(f"\n  Distribution 'label' (top 10) :")
print(df_qc['label'].value_counts(dropna=False).head(10).to_string())

# ── Sauvegarde ───────────────────────────────────────────────────────────────
save_csv(df_qc, OUT_QC_CSV, "QUALITY CONTROL")
preview(df_qc)

═══════════════════════════════════════════════════════
ÉTAPE 8 — Fichier Contrôle Qualité
═══════════════════════════════════════════════════════

  Nombre total de textes : 24800

  Distribution 'decision' :
decision
garder      19977
corriger     4476
rejeter       347

  Distribution 'label' (top 10) :
label
opinion             5203
soutien             5062
food                2229
inclassable         2067
question_general    2005
politics            1854
entertainment       1734
business            1493
sports               996
medical              829
  ✅ [QUALITY CONTROL] Sauvegardé : enfin/quality_control.csv  (24800 lignes × 20 colonnes)
   Shape : (24800, 20)  |  Colonnes : ['entry_id', 'text', 'text_normalized', 'label', 'decision', 'source_type', 'channel_name', 'domain', 'published_at', 'language', 'script', 'darija_type', 'has_codeswitching', 'has_arabizi', 'det_confidence', 'darija_ara_score', 'darija_lat_score', 'msa_score', 'n_words', 'n_chars']


,entry_id,text,text_normalized,label,decision,source_type,channel_name,domain,published_at,language,script,darija_type,has_codeswitching,has_arabizi,det_confidence,darija_ara_score,darija_lat_score,msa_score,n_words,n_chars
0,darija_050244,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,food,garder,youtube_comment,Cuisine Halima Filali,food,1.770682e+09,darija,arabic,darija_arabic,False,False,0.600,0.2000,0.0000,0.0,10.0,58.0
1,darija_082567,مالكم.درقو.فالبنت.واش.حنا.خصنا,مالكم.درقو.فالبنت.واش.حنا.خصنا,inclassable,corriger,youtube_comment,Simo Stars,food,1.765325e+09,darija,arabic,darija_arabic,False,False,0.500,0.3333,0.0000,0.0,1.0,30.0
2,darija_006392,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,business,garder,youtube_comment,Youssef Akalal,business,1.678406e+09,darija,latin,darija_latin,False,True,0.625,0.0000,0.3333,0.0,12.0,69.0


---
## ÉTAPE 9 — Fichier Modélisation

**Contient : uniquement les textes `garder` et `corriger`** — les `rejeter` sont exclus.  
**La colonne `decision` est supprimée** (non nécessaire pour l'entraînement).

### Colonnes du fichier `modelisation.csv` :

| Colonne | Rôle dans la modélisation |
|---|---|
| `entry_id` | Identifiant unique |
| `text` | Texte brut original |
| `text_normalized` | Texte sans emojis → input principal du modèle |
| `label` | **Variable cible (y)** |
| `source_type` | Feature catégorielle (provenance) |
| `script` | Feature catégorielle (arabic/latin/mixed) |
| `darija_type` | Feature catégorielle (type Darija) |
| `has_codeswitching` | Feature binaire |
| `has_arabizi` | Feature binaire |
| `n_words` | Feature numérique |
| `n_chars` | Feature numérique |
| `n_arabic_chars` | Feature numérique |
| `n_latin_chars` | Feature numérique |
| `darija_ara_score` | Feature numérique (score linguistique) |
| `darija_lat_score` | Feature numérique (score linguistique) |
| `msa_score` | Feature numérique (score MSA) |

In [11]:
print("═" * 55)
print("ÉTAPE 9 — Fichier Modélisation")
print("═" * 55)

# ── Filtrage : garder + corriger uniquement (exclusion des rejeter) ───────────
df_model_base = df_enriched[
    df_enriched['decision'].str.lower().isin(['garder', 'corriger'])
].copy()

n_rejete = len(df_enriched) - len(df_model_base)
print(f"  Textes 'rejeter' exclus : {n_rejete}")
print(f"  Textes retenus          : {len(df_model_base)}")

# ── Colonnes pour la modélisation ────────────────────────────────────────────
MODEL_COLS = [
    # ── Identité & textes ────────────────────────────────
    "entry_id",
    "text",              # texte brut original
    "text_normalized",   # texte sans emojis → input modèle
    "label",             # variable cible (y)
    # ── Features catégorielles ───────────────────────────
    "source_type",
    "script",
    "darija_type",
    "has_codeswitching",
    "has_arabizi",
    # ── Features numériques ──────────────────────────────
    "n_words",
    "n_chars",
    "n_arabic_chars",
    "n_latin_chars",
    "darija_ara_score",
    "darija_lat_score",
    "msa_score",
    # NOTE : 'decision' est VOLONTAIREMENT exclue
]

model_cols_present = [c for c in MODEL_COLS if c in df_model_base.columns]
model_cols_absent  = [c for c in MODEL_COLS if c not in df_model_base.columns]

if model_cols_absent:
    print(f"  ⚠️  Colonnes absentes (non enrichies) : {model_cols_absent}")

df_model = df_model_base[model_cols_present].copy()

# ── Statistiques ─────────────────────────────────────────────────────────────
print(f"\n  Distribution 'label' (variable cible) :")
label_dist = df_model['label'].value_counts(dropna=False)
print(label_dist.to_string())

print(f"\n  Labels vides (NaN) : {df_model['label'].isna().sum()}")
print(f"  Colonnes finales ({len(df_model.columns)}) : {list(df_model.columns)}")

# ── Sauvegarde ───────────────────────────────────────────────────────────────
save_csv(df_model, OUT_MODEL_CSV, "MODÉLISATION")
preview(df_model)

═══════════════════════════════════════════════════════
ÉTAPE 9 — Fichier Modélisation
═══════════════════════════════════════════════════════
  Textes 'rejeter' exclus : 347
  Textes retenus          : 24453

  Distribution 'label' (variable cible) :
label
opinion             5170
soutien             4990
food                2200
inclassable         2002
question_general    1979
politics            1834
entertainment       1712
business            1475
sports               987
medical              799
religion             721
publicite            573
opinion                2
business               2
NaN                    2
 inclassable           1
 food                  1
                       1
questio_general        1
inclassabe             1

  Labels vides (NaN) : 2
  Colonnes finales (16) : ['entry_id', 'text', 'text_normalized', 'label', 'source_type', 'script', 'darija_type', 'has_codeswitching', 'has_arabizi', 'n_words', 'n_chars', 'n_arabic_chars', 'n_latin_chars', 'darija_

,entry_id,text,text_normalized,label,source_type,script,darija_type,has_codeswitching,has_arabizi,n_words,n_chars,n_arabic_chars,n_latin_chars,darija_ara_score,darija_lat_score,msa_score
0,darija_050244,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,واش ملي نعمروهم في البواطة نعمل في الكنجلاطور ...,food,youtube_comment,arabic,darija_arabic,False,False,10.0,58.0,49.0,0.0,0.2000,0.0000,0.0
1,darija_082567,مالكم.درقو.فالبنت.واش.حنا.خصنا,مالكم.درقو.فالبنت.واش.حنا.خصنا,inclassable,youtube_comment,arabic,darija_arabic,False,False,1.0,30.0,25.0,0.0,0.3333,0.0000,0.0
2,darija_006392,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,Jay dyal merch 3ad chi dawra nadiya dyal ecomm...,business,youtube_comment,latin,darija_latin,False,True,12.0,69.0,0.0,57.0,0.0000,0.3333,0.0


---
## ✅ Récapitulatif final

In [12]:
print("═" * 70)
print("RÉCAPITULATIF — TOUS LES FICHIERS GÉNÉRÉS")
print("═" * 70)

files = [
    (OUT_JSON1_CLEAN, "JSON 1 nettoyé"),
    (OUT_JSON2_CLEAN, "JSON 2 nettoyé (garder_gemini / garder_manuel / corriger)"),
    (OUT_JSON3_CLEAN, "JSON 3 nettoyé"),
    (OUT_JSON4_CLEAN, "JSON 4 nettoyé"),
    (OUT_MERGED_CSV,  "Fusion des 4 JSON (.csv)"),
    (OUT_MERGED_JSON, "Fusion des 4 JSON (.json)"),
    (OUT_QC_CSV,      "✅ Contrôle Qualité — TOUS les textes (garder+corriger+rejeter)"),
    (OUT_MODEL_CSV,   "✅ Modélisation — garder+corriger uniquement, sans 'decision'"),
]

for path, desc in files:
    p    = Path(path)
    size = f"{p.stat().st_size / 1024:.1f} KB" if p.exists() else "NON GÉNÉRÉ"
    icon = '✅' if p.exists() else '❌'
    print(f"  {icon}  {Path(path).name:<30} {size:<12}  {desc}")

print()
print("─" * 70)
print(f"📊 Contrôle Qualité : {len(df_qc):>6} lignes × {len(df_qc.columns)} colonnes")
print(f"🤖 Modélisation     : {len(df_model):>6} lignes × {len(df_model.columns)} colonnes")
print(f"   dont rejeter exclus : {n_rejete} lignes")
print("─" * 70)
print()
print("Colonnes Contrôle Qualité :")
print(f"  {list(df_qc.columns)}")
print()
print("Colonnes Modélisation :")
print(f"  {list(df_model.columns)}")

══════════════════════════════════════════════════════════════════════
RÉCAPITULATIF — TOUS LES FICHIERS GÉNÉRÉS
══════════════════════════════════════════════════════════════════════
  ✅  json1_clean.csv                114.1 KB      JSON 1 nettoyé
  ✅  json2_clean.csv                146.6 KB      JSON 2 nettoyé (garder_gemini / garder_manuel / corriger)
  ✅  json3_clean.csv                1974.8 KB     JSON 3 nettoyé
  ✅  json4_clean.csv                1197.0 KB     JSON 4 nettoyé
  ✅  merged.csv                     3432.1 KB     Fusion des 4 JSON (.csv)
  ✅  merged.json                    5416.4 KB     Fusion des 4 JSON (.json)
  ✅  quality_control.csv            8923.1 KB     ✅ Contrôle Qualité — TOUS les textes (garder+corriger+rejeter)
  ✅  modelisation.csv               7702.0 KB     ✅ Modélisation — garder+corriger uniquement, sans 'decision'

──────────────────────────────────────────────────────────────────────
📊 Contrôle Qualité :  24800 lignes × 20 colonnes
🤖 Modélisation   